In [48]:
import pandas as pd

url1 = "heart_stroke_train_dataset.csv"
url2 = "heart_stroke_test_dataset.csv"

train_df = pd.read_csv(url1)
test_df = pd.read_csv(url2)

In [49]:
train_df.head()

,ID,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,846,Female,48.0,0,0,Yes,Private,Urban,69.21,33.1,never smoked,0
1,3745,Male,15.0,0,0,No,Private,Rural,122.25,21.0,never smoked,0
2,4184,Female,67.0,0,0,Yes,Self-employed,Rural,110.42,24.9,never smoked,0
3,3410,Male,44.0,0,0,Yes,Private,Urban,65.41,24.8,smokes,0
4,285,Male,14.0,0,0,No,Govt_job,Urban,82.34,31.6,Unknown,0


In [50]:
print(train_df.shape)
print(test_df.shape)

(4088, 12)
(1022, 11)


In [51]:
train_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4088 entries, 0 to 4087
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ID                 4088 non-null   int64  
 1   gender             4088 non-null   object 
 2   age                4088 non-null   float64
 3   hypertension       4088 non-null   int64  
 4   heart_disease      4088 non-null   int64  
 5   ever_married       4088 non-null   object 
 6   work_type          4088 non-null   object 
 7   Residence_type     4088 non-null   object 
 8   avg_glucose_level  4088 non-null   float64
 9   bmi                3918 non-null   float64
 10  smoking_status     4088 non-null   object 
 11  stroke             4088 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 383.4+ KB


In [52]:
train_df.describe()

,ID,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,4088.000000,4088.000000,4088.000000,4088.000000,4088.000000,3918.000000,4088.000000
mean,2539.784002,43.353288,0.097114,0.054061,106.317167,28.922180,0.048679
std,1484.069131,22.596816,0.296148,0.226165,45.259652,7.928378,0.215223
min,1.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,1250.750000,26.000000,0.000000,0.000000,77.312500,23.600000,0.000000
50%,2525.000000,45.000000,0.000000,0.000000,91.945000,28.000000,0.000000
75%,3844.250000,61.000000,0.000000,0.000000,114.197500,33.100000,0.000000
max,5110.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


In [53]:

train_df.isna().sum()

ID                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  170
smoking_status         0
stroke                 0
dtype: int64

In [54]:
print(train_df.isna().sum()/len(train_df))

ID                   0.000000
gender               0.000000
age                  0.000000
hypertension         0.000000
heart_disease        0.000000
ever_married         0.000000
work_type            0.000000
Residence_type       0.000000
avg_glucose_level    0.000000
bmi                  0.041585
smoking_status       0.000000
stroke               0.000000
dtype: float64


In [55]:
df=train_df.drop_duplicates(keep=False)
print(df.shape) #no duplicates found

(4088, 12)


Median Imputation on BMI

In [56]:

train_df['bmi'] = train_df['bmi'].fillna(train_df['bmi'].median())
test_df['bmi'] = test_df['bmi'].fillna(train_df['bmi'].median())


Encoding

In [57]:

train_df_en = pd.get_dummies(train_df,drop_first=True)

print(train_df_en.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4088 entries, 0 to 4087
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   ID                              4088 non-null   int64  
 1   age                             4088 non-null   float64
 2   hypertension                    4088 non-null   int64  
 3   heart_disease                   4088 non-null   int64  
 4   avg_glucose_level               4088 non-null   float64
 5   bmi                             4088 non-null   float64
 6   stroke                          4088 non-null   int64  
 7   gender_Male                     4088 non-null   bool   
 8   gender_Other                    4088 non-null   bool   
 9   ever_married_Yes                4088 non-null   bool   
 10  work_type_Never_worked          4088 non-null   bool   
 11  work_type_Private               4088 non-null   bool   
 12  work_type_Self-employed         40

In [58]:

from sklearn.model_selection import train_test_split



X = train_df_en.drop(columns=['stroke','ID'],axis=1)
y = train_df_en['stroke']

X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=42,test_size=0.2)

X_train.shape,X_test.shape,y_train.shape,y_test.shape

((3270, 16), (818, 16), (3270,), (818,))

In [59]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaler = scaler.fit_transform(X_train)

X_test_scaler = scaler.transform(X_test)

Train model

In [60]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(solver = "liblinear",random_state=42,max_iter=1000,class_weight="balanced")
log_reg.fit(X_train_scaler,y_train)


LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42,
                   solver='liblinear')

After adding class_weigt="balanced",Increasing the penalty for the minority class makes the optimizer treat those errors as more costly. As a result, the model shifts its decision boundary to predict more minority-class samples, which usually increases recall for stroke cases at the expense of some additional false positives and lower overall accuracy

In [61]:

y_pred = log_reg.predict(X_test_scaler)

In [62]:

log_reg.intercept_

array([-1.17952987])

In [63]:
log_reg.classes_

array([0, 1], dtype=int64)

In [64]:
log_reg.intercept_

array([-1.17952987])

In [65]:

log_reg.coef_

array([[ 1.99636634,  0.17869742,  0.04356026,  0.13346124,  0.16118044,
        -0.00429961, -0.05519225, -0.12185116, -0.15373393, -0.05097454,
        -0.17286587,  0.43511843,  0.11848728, -0.01007608, -0.13987362,
         0.13134386]])

In [80]:
#Predict probabilities for the positive class (class 1)
y_prob = log_reg.predict_proba(X_test_scaler)[:,1]
y_prob

array([0.77494635, 0.02144844, 0.02791305, 0.91589219, 0.74225311,
       0.29463864, 0.4535415 , 0.59255136, 0.47835057, 0.03508881,
       0.10419694, 0.05246965, 0.82218233, 0.95248126, 0.03338525,
       0.11323002, 0.56263479, 0.03088874, 0.62234629, 0.10880021,
       0.04211541, 0.08241656, 0.51003015, 0.40242619, 0.14816794,
       0.26504322, 0.09744636, 0.04642755, 0.81579524, 0.8199554 ,
       0.10739182, 0.14894713, 0.08590799, 0.55822058, 0.06630812,
       0.88533766, 0.02915222, 0.85872548, 0.47249579, 0.54270069,
       0.75262558, 0.26724296, 0.32623621, 0.08009835, 0.05239106,
       0.84010852, 0.07574342, 0.9146952 , 0.61942649, 0.5863453 ,
       0.01959017, 0.06863401, 0.0420017 , 0.50291737, 0.67142511,
       0.59093601, 0.05545934, 0.65066477, 0.03784436, 0.18240864,
       0.06263611, 0.02784567, 0.78774457, 0.29305453, 0.06926193,
       0.04058863, 0.66172837, 0.02350931, 0.77966311, 0.07933912,
       0.84480292, 0.26365245, 0.77424644, 0.87368642, 0.91247

In binary classification, predict_proba() returns probabilities for both classes: column 0 is P(y=0) and column 1 is P(y=1). We usually extract [:,1] because metrics such as ROC-AUC, log loss, precision-recall analysis, and threshold tuning are defined using the probability of the positive class, which in this stroke problem is stroke = 1. The probability of class 0 is redundant because P(0) = 1 - P(1)

Model Prediction

In [67]:

log_reg.score(X_test_scaler,y_test)

0.7273838630806846

In [68]:
from sklearn.metrics import accuracy_score,classification_report

y_pred = log_reg.predict(X_test_scaler)
acc = accuracy_score(y_test, y_pred)
print(acc)

0.7273838630806846


In [69]:
y.value_counts()

stroke
0    3889
1     199
Name: count, dtype: int64

Model Evaluation:cost function

In [77]:

from sklearn.metrics import log_loss

# Calculate log loss
log_loss_value = log_loss(y_test, y_prob)
print(f"Log Loss : {log_loss_value:.4f}")

Log Loss : 0.5366


In [71]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.72      0.84       781
           1       0.12      0.81      0.21        37

    accuracy                           0.73       818
   macro avg       0.55      0.77      0.52       818
weighted avg       0.95      0.73      0.81       818



The balanced logistic regression model is able to detect most stroke cases (high recall = 0.81), which is desirable for a medical screening application. However, it produces many false positives (low precision = 0.12), leading to a reduction in overall accuracy to 73%. The log loss of 0.50 indicates that the predicted probabilities are reasonably calibrated and the model is not making many extremely overconfident errors.